# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis: one content page** (`content_id`), a pseudonymized page belonging to one of 32
pseudonymized clients (`client_id`).

**Time window:** every activity metric is a trailing **90-day window ending at the export date**.
The 30-day comparison columns split that same window into a "last 30 days" and "prior 30 days"
sub-window, used only to compute the trend fields. `content_age_days` / `days_since_last_update`
are point-in-time counters as of export, not windowed sums. Verified below.


In [1]:
import os, sys
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

dup = df["content_id"].duplicated().sum()
print("Duplicate content_id rows (should be 0 if the grain holds):", dup)

print("\ncontent_age_days range:", df["content_age_days"].min(), "-", df["content_age_days"].max())
print("(the docs promise every row is >= 90 days old in this slice -- confirmed above)")

window_check = (df["impressions_last_30d"] + df["impressions_prev_30d"] <= df["impressions_90d"] + 1).mean()
print(f"\nShare of rows where last30 + prev30 <= 90d total "
      f"(sanity check on the two sub-windows nesting inside the 90d window): {window_check:.3f}")


Duplicate content_id rows (should be 0 if the grain holds): 0

content_age_days range: 90 - 564
(the docs promise every row is >= 90 days old in this slice -- confirmed above)

Share of rows where last30 + prev30 <= 90d total (sanity check on the two sub-windows nesting inside the 90d window): 1.000


## 2. Fields: feature / label / context / excluded

| Bucket | Fields |
|---|---|
| **Feature** | `search_volume, competition, cpc, word_count, char_count, log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d, days_with_impressions, days_with_sessions, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct` (numeric) + `competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, position_tier` (categorical) -- exactly `MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES` in `scripts/ml_utils.py`. |
| **Label / proxy** | `trend_direction` (the label IS `trend_direction == "down"`) and `trend_pct` (the number that bucket is computed from). Neither is ever a feature. |
| **Context** (grouping / joining / splitting only) | `content_id` (unique row key), `client_id` (used for the client-holdout split -- never a feature). |
| **Excluded, with why** | `provider_used`, `model_used` -- which LLM/provider authored the article; the data dictionary marks both "not a model feature," and it's an authoring detail, not a search signal. `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` -- these sub-windows are exactly what `trend_pct` is built from, so they are label-adjacent and excluded even though they aren't the label itself. |


In [2]:
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

feature_cols = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
label_cols = ["trend_direction", "trend_pct"]
context_cols = ["content_id", "client_id"]

overlap = set(feature_cols) & set(label_cols)
print("Feature/label overlap (must be empty):", overlap)

missing_from_raw = sorted(set(feature_cols) - set(df.columns))
print("Feature columns missing from the raw CSV (expected: the log_* ones, built in the prep step):")
print(" ", missing_from_raw)

print("\nclient_id distinct values:", df["client_id"].nunique(), "(context only, used for grouped splits)")


Feature/label overlap (must be empty): set()
Feature columns missing from the raw CSV (expected: the log_* ones, built in the prep step):
  ['log_ai_sessions_90d', 'log_clicks_90d', 'log_impressions_90d', 'log_sessions_90d']

client_id distinct values: 32 (context only, used for grouped splits)


## 3. Verify it with queries (grain, counts, missing values, windows)

Checking the data dictionary's warning that missingness follows `content_type`, not chance:


In [3]:
missing_by_type = df.groupby("content_type")[["search_volume", "word_count"]].apply(lambda g: g.isna().mean())
print(missing_by_type.round(3))
print("\n-> missingness is NOT uniform across content_type -- a blind fillna(0) would silently")
print("   encode content_type into the features. Confirmed: has_-style flags or per-type handling")
print("   are needed here, not a blind fill (this is what scripts/01_prepare_features.py actually does).")


                    search_volume  word_count
content_type                                 
comparison article          0.000       0.000
feedly article              1.000       0.000
keyword article             0.014       0.283

-> missingness is NOT uniform across content_type -- a blind fillna(0) would silently
   encode content_type into the features. Confirmed: has_-style flags or per-type handling
   are needed here, not a blind fill (this is what scripts/01_prepare_features.py actually does).


## 4. Data limits

- **Proxy label, not a future outcome.** `is_declining_label` describes the CURRENT window, not
  what happens next -- it can say which pages look weakest today, not whether review would help.
- **Unbalanced client coverage.** 32 clients share 30,000 pages very unevenly (checked below) -- a
  client with many pages can dominate any pattern that isn't checked per-client.
- **No causal claim possible.** This data alone cannot say a refresh caused a recovery -- only an
  experiment could. Every output here stays decision-support, never causal.
- **`avg_position == 0` means "no position data," not position zero** -- must be filtered or
  flagged before use, never treated as a great ranking.
- **Rate columns are already x100 percentages** (`ctr`, `engagement_rate`, `scroll_rate`,
  `ai_traffic_pct`, `trend_pct`) -- `ctr = 0.76` means 0.76%, not 76%. Getting this wrong silently
  breaks every downstream threshold.


In [4]:
page_counts = df.groupby("client_id").size()
print("Pages per client -- min/median/max:", page_counts.min(), int(page_counts.median()), page_counts.max())

zero_position = (df["avg_position"] == 0).sum()
print(f"\navg_position == 0 (no position data, not rank 0): {zero_position} rows ({zero_position/len(df):.1%})")

print(f"\nMean ctr in the raw column: {df['ctr'].mean():.3f} "
      "-- this is already a percentage (x100), not a 0-1 fraction.")


Pages per client -- min/median/max: 3 567 7008

avg_position == 0 (no position data, not rank 0): 1205 rows (4.0%)

Mean ctr in the raw column: 0.511 -- this is already a percentage (x100), not a 0-1 fraction.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.